In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from datasets.drive_dataset import DriveDataset
from models.mini_unet import MiniUNet
from utils.losses import BCETverskyLoss


def train():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", device)

    train_dataset = DriveDataset(
        root_dir=r"E:\CS\PostG\个人\unet_learn\DRIVE\training",
        image_size=256
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=2,
        shuffle=True
    )

    model = MiniUNet(in_channels=3, num_classes=1).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    epochs = 10

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0

        for images, masks in train_loader:
            images = images.to(device)
            masks = masks.to(device)

            logits = model(images)
            loss = criterion(logits, masks)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)

        print(f"Epoch [{epoch+1}/{epochs}] Loss: {avg_loss:.4f}")

    torch.save(model.state_dict(), "mini_unet_drive.pth")
    print("model saved to mini_unet_drive.pth")


if __name__ == "__main__":
    train()

device: cuda
input: torch.Size([2, 3, 256, 256])
x1 / inc: torch.Size([2, 64, 256, 256])
x2 / down1: torch.Size([2, 128, 128, 128])
x3 / down2: torch.Size([2, 256, 64, 64])
x4 / bottleneck: torch.Size([2, 512, 32, 32])
up1: torch.Size([2, 256, 64, 64])
up2: torch.Size([2, 128, 128, 128])
up3: torch.Size([2, 64, 256, 256])
out: torch.Size([2, 1, 256, 256])
input: torch.Size([2, 3, 256, 256])
x1 / inc: torch.Size([2, 64, 256, 256])
x2 / down1: torch.Size([2, 128, 128, 128])
x3 / down2: torch.Size([2, 256, 64, 64])
x4 / bottleneck: torch.Size([2, 512, 32, 32])
up1: torch.Size([2, 256, 64, 64])
up2: torch.Size([2, 128, 128, 128])
up3: torch.Size([2, 64, 256, 256])
out: torch.Size([2, 1, 256, 256])
input: torch.Size([2, 3, 256, 256])
x1 / inc: torch.Size([2, 64, 256, 256])
x2 / down1: torch.Size([2, 128, 128, 128])
x3 / down2: torch.Size([2, 256, 64, 64])
x4 / bottleneck: torch.Size([2, 512, 32, 32])
up1: torch.Size([2, 256, 64, 64])
up2: torch.Size([2, 128, 128, 128])
up3: torch.Size([2, 6

In [2]:
import torch
from torch.utils.data import DataLoader, random_split

from datasets.drive_dataset import DriveDataset
from models.mini_unet import MiniUNet
# from utils.losses import BCEDiceLoss
from utils.losses import BCETverskyLoss
from utils.metrics import binary_iou

from pathlib import Path
import csv


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    total_iou = 0.0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        logits = model(images)
        loss = criterion(logits, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_iou += binary_iou(logits, masks)

    return total_loss / len(loader), total_iou / len(loader)


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_iou = 0.0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        logits = model(images)
        loss = criterion(logits, masks)

        total_loss += loss.item()
        total_iou += binary_iou(logits, masks)

    return total_loss / len(loader), total_iou / len(loader)


def main():

    # ====== log / checkpoint ======
    Path("logs").mkdir(exist_ok=True)
    Path("checkpoints").mkdir(exist_ok=True)

    log_path = Path("logs/train_log_mini_unet_tversky.csv")
    best_model_path = Path("checkpoints/best_mini_unet_drive_tversky.pth")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", device)

    # ====== dataset ======
    dataset = DriveDataset(
        root_dir=r"E:\CS\PostG\个人\unet_learn\DRIVE\training",
        image_size=256
    )

    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size

    train_dataset, val_dataset = random_split(
        dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )

    train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False)

    # ====== model ======
    model = MiniUNet(in_channels=3, num_classes=1).to(device)

    #criterion = BCEDiceLoss(bce_weight=0.5, dice_weight=0.5)
    criterion = BCETverskyLoss(
        bce_weight=0.3,
        tversky_weight=0.7
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    # ====== CSV ======
    with open(log_path, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "epoch",
            "train_loss",
            "train_iou",
            "val_loss",
            "val_iou",
            "best_val_iou"
        ])

        epochs = 50
        best_val_iou = 0.0

        for epoch in range(epochs):

            train_loss, train_iou = train_one_epoch(
                model, train_loader, criterion, optimizer, device
            )

            val_loss, val_iou = validate(
                model, val_loader, criterion, device
            )

            if val_iou > best_val_iou:
                best_val_iou = val_iou
                torch.save(model.state_dict(), best_model_path)
                save_msg = " | saved best"
            else:
                save_msg = ""

            writer.writerow([
                epoch + 1,
                train_loss,
                train_iou,
                val_loss,
                val_iou,
                best_val_iou
            ])

            print(
                f"Epoch [{epoch+1}/{epochs}] "
                f"Train Loss: {train_loss:.4f} "
                f"Train IoU: {train_iou:.4f} "
                f"Val Loss: {val_loss:.4f} "
                f"Val IoU: {val_iou:.4f}"
                f"{save_msg}"
            )

    print("training finished")
    print("best val iou:", best_val_iou)
    print("log saved to:", log_path)
    print("best model saved to:", best_model_path)


if __name__ == "__main__":
    main()

device: cuda
Epoch [1/50] Train Loss: 0.7118 Train IoU: 0.1254 Val Loss: 0.7671 Val IoU: 0.0000 | saved best
Epoch [2/50] Train Loss: 0.6530 Train IoU: 0.2019 Val Loss: 0.7690 Val IoU: 0.0523 | saved best
Epoch [3/50] Train Loss: 0.6060 Train IoU: 0.2889 Val Loss: 0.7522 Val IoU: 0.0034
Epoch [4/50] Train Loss: 0.5778 Train IoU: 0.3308 Val Loss: 0.7207 Val IoU: 0.0168
Epoch [5/50] Train Loss: 0.5523 Train IoU: 0.3640 Val Loss: 0.6791 Val IoU: 0.0957 | saved best
Epoch [6/50] Train Loss: 0.5298 Train IoU: 0.4073 Val Loss: 0.6306 Val IoU: 0.2223 | saved best
Epoch [7/50] Train Loss: 0.5156 Train IoU: 0.4217 Val Loss: 0.5948 Val IoU: 0.2852 | saved best
Epoch [8/50] Train Loss: 0.5027 Train IoU: 0.4402 Val Loss: 0.5710 Val IoU: 0.3099 | saved best
Epoch [9/50] Train Loss: 0.4914 Train IoU: 0.4582 Val Loss: 0.5475 Val IoU: 0.3722 | saved best
Epoch [10/50] Train Loss: 0.4838 Train IoU: 0.4786 Val Loss: 0.5345 Val IoU: 0.3902 | saved best
Epoch [11/50] Train Loss: 0.4747 Train IoU: 0.4782 V

In [1]:
import torch
# from torch.utils.data import DataLoader, random_split

from datasets.drive_dataset import DriveDataset
# help(DriveDataset.__init__)
#from models.mini_unet import MiniUNet
from models.resunet import ResUNet
#from utils.losses import BCEDiceLoss
from utils.losses import BCETverskyLoss
from utils.metrics import binary_iou

from pathlib import Path
import csv

from torch.utils.data import DataLoader, Subset


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()

    total_loss = 0.0
    total_iou = 0.0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        logits = model(images)
        loss = criterion(logits, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_iou += binary_iou(logits, masks)

    return total_loss / len(loader), total_iou / len(loader)


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()

    total_loss = 0.0
    total_iou = 0.0

    for images, masks in loader:
        images = images.to(device)
        masks = masks.to(device)

        logits = model(images)
        loss = criterion(logits, masks)

        total_loss += loss.item()
        total_iou += binary_iou(logits, masks)

    return total_loss / len(loader), total_iou / len(loader)


def main():

    # ====== log / checkpoint ======
    Path("logs").mkdir(exist_ok=True)
    Path("checkpoints").mkdir(exist_ok=True)

    # log_path = Path("logs/train_resunet_log4.csv")
    # best_model_path = Path("checkpoints/best_resunet_drive4.pth")
    log_path = Path("logs/train_resunet_se_tversky_aug_log.csv")
    best_model_path = Path("checkpoints/best_resunet_se_tversky_aug_drive.pth")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("device:", device)

    # ====== dataset ======
    # dataset = DriveDataset(
    #     root_dir=r"E:\CS\PostG\个人\unet_learn\DRIVE\training",
    #     image_size=256
    # )

    # train_size = int(0.8 * len(dataset))
    # val_size = len(dataset) - train_size

    # train_dataset, val_dataset = random_split(
    #     dataset,
    #     [train_size, val_size],
    #     generator=torch.Generator().manual_seed(42)
    # )

    # train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
    # val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False)

    train_base_dataset = DriveDataset(
        root_dir=r"E:\CS\PostG\个人\unet_learn\DRIVE\training",
        image_size=256,
        is_train=True
    )   

    val_base_dataset = DriveDataset(
        root_dir=r"E:\CS\PostG\个人\unet_learn\DRIVE\training",
        image_size=256,
        is_train=False
    )

    dataset_size = len(train_base_dataset)
    train_size = int(0.8 * dataset_size)

    indices = torch.randperm(
        dataset_size,
        generator=torch.Generator().manual_seed(42)
    ).tolist()

    train_indices = indices[:train_size]
    val_indices = indices[train_size:]

    train_dataset = Subset(train_base_dataset, train_indices)
    val_dataset = Subset(val_base_dataset, val_indices)

    train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False)


    # ====== model ======
    #model = MiniUNet(in_channels=3, num_classes=1).to(device)
    model = ResUNet(num_classes=1).to(device)

    # criterion = BCEDiceLoss(bce_weight=0.5, dice_weight=0.5)
    criterion = BCETverskyLoss(
        bce_weight=0.3,
        tversky_weight=0.7
    )

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    # ====== CSV ======
    with open(log_path, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([
            "epoch",
            "train_loss",
            "train_iou",
            "val_loss",
            "val_iou",
            "best_val_iou"
        ])

        epochs = 50
        best_val_iou = 0.0

        for epoch in range(epochs):

            train_loss, train_iou = train_one_epoch(
                model, train_loader, criterion, optimizer, device
            )

            val_loss, val_iou = validate(
                model, val_loader, criterion, device
            )

            if val_iou > best_val_iou:
                best_val_iou = val_iou
                torch.save(model.state_dict(), best_model_path)
                save_msg = " | saved best"
            else:
                save_msg = ""

            writer.writerow([
                epoch + 1,
                train_loss,
                train_iou,
                val_loss,
                val_iou,
                best_val_iou
            ])

            print(
                f"Epoch [{epoch+1}/{epochs}] "
                f"Train Loss: {train_loss:.4f} "
                f"Train IoU: {train_iou:.4f} "
                f"Val Loss: {val_loss:.4f} "
                f"Val IoU: {val_iou:.4f}"
                f"{save_msg}"
            )

    print("training finished")
    print("best val iou:", best_val_iou)
    print("log saved to:", log_path)
    print("best model saved to:", best_model_path)


if __name__ == "__main__":
    main()

device: cuda
Epoch [1/50] Train Loss: 0.6919 Train IoU: 0.1790 Val Loss: 0.7642 Val IoU: 0.0000 | saved best
Epoch [2/50] Train Loss: 0.6172 Train IoU: 0.2715 Val Loss: 0.7587 Val IoU: 0.0000
Epoch [3/50] Train Loss: 0.5612 Train IoU: 0.3310 Val Loss: 0.7384 Val IoU: 0.0003 | saved best
Epoch [4/50] Train Loss: 0.5249 Train IoU: 0.3750 Val Loss: 0.7307 Val IoU: 0.0275 | saved best
Epoch [5/50] Train Loss: 0.5014 Train IoU: 0.3989 Val Loss: 0.6520 Val IoU: 0.2049 | saved best
Epoch [6/50] Train Loss: 0.4794 Train IoU: 0.4285 Val Loss: 0.5968 Val IoU: 0.2709 | saved best
Epoch [7/50] Train Loss: 0.4601 Train IoU: 0.4490 Val Loss: 0.5309 Val IoU: 0.4376 | saved best
Epoch [8/50] Train Loss: 0.4497 Train IoU: 0.4603 Val Loss: 0.4961 Val IoU: 0.4510 | saved best
Epoch [9/50] Train Loss: 0.4408 Train IoU: 0.4701 Val Loss: 0.4738 Val IoU: 0.4638 | saved best
Epoch [10/50] Train Loss: 0.4301 Train IoU: 0.4808 Val Loss: 0.4526 Val IoU: 0.4762 | saved best
Epoch [11/50] Train Loss: 0.4254 Train 